# GTEx k-means clustering of gene expression data using different methods

The clustering ensemble itself is run separately by the `kmeans_clustering_gtex` rule (`scripts/gtex/kmeans_clustering.py`); this notebook loads those cached per-method results and renders the ARI comparison figures. It evaluates how well different dimensionality reduction methods preserve biological tissue structure in GTEx gene expression data, comparing raw RNA-Seq, CLAMP base and CLAMPfull, PCA, ICA, NMF, PLIER, Flashier, MOFA-FLEX, and GenomicSuperSignature by Adjusted Rand Index (ARI) against ground-truth tissue labels.

💡 **Environment:** `clamp-analyses`

## Libraries

In [ ]:
import pickle
import re
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from pyprojroot.here import here


## Settings

In [ ]:
with open(here("config.yaml"), "r") as f:
    _cfg = yaml.safe_load(f)

MODEL_COLORS = _cfg["MODEL_COLORS"]
RNASEQ_FRACTION_COLORS = _cfg["RNASEQ_FRACTION_COLORS"]

def get_model_color(model_name):
    return MODEL_COLORS.get(model_name, "#333333")

def get_fraction_color(fraction_label):
    return RNASEQ_FRACTION_COLORS.get(str(fraction_label), "#333333")


# Load cached results

In [ ]:
with open(here("workflow/config/gtex.yaml"), "r") as f:
    GTEX_CONFIG = yaml.safe_load(f)["gtex"]

KMEANS_CACHE_DIR = Path(here(f"{GTEX_CONFIG['paths']['biology']}/00_kmeans_clustering/kmeans_results"))

METHOD_CACHE_KEYS = {
    "CLAMP_base": "gtex_CLAMP_base_kmeans",
    "CLAMP_full": "gtex_CLAMPfull_kmeans",
    "PLIER": "gtex_PLIER_kmeans",
    "GenomicSuperSignature": "gtex_GenomicSuperSignature_kmeans",
    "flashier": "gtex_flashier_kmeans",
    "mofabc": "gtex_mofabc_kmeans",
    "pca": "gtex_pca_kmeans",
    "ica": "gtex_ica_kmeans",
    "nmf": "gtex_nmf_kmeans",
}

for suffix, cache_key in METHOD_CACHE_KEYS.items():
    with open(KMEANS_CACHE_DIR / f"{cache_key}.pkl", "rb") as f:
        results = pickle.load(f)
    globals()[f"best_k_gtex_{suffix}_kmeans_results_df"] = results["best_k_results_df"]

with open(KMEANS_CACHE_DIR / "gtex_rnaseq_topvar_tpm_variance_results.pkl", "rb") as f:
    rnaseq_topvar_results = pickle.load(f)

best_mean_ari = -1
best_run_label = None
for run_label, results in rnaseq_topvar_results.items():
    ari = results["scaled_ari"]
    if ari > best_mean_ari:
        best_mean_ari = ari
        best_run_label = run_label

print(f"Best RNA-Seq configuration: {best_run_label}")
print(f"  Best k: {rnaseq_topvar_results[best_run_label]['best_k']}")
print(f"  Mean ARI: {best_mean_ari:.3f}")

best_k_gtex_rnaseq_kmeans_results_df = rnaseq_topvar_results[best_run_label]["best_k_results_df"]

best_k_rnaseq_results = {}
for run_label, results in rnaseq_topvar_results.items():
    pct = int(run_label.split("_")[1].replace("pct", ""))
    frac = pct / 100.0
    df = results["best_k_results_df"].copy()
    df["frac"] = frac
    best_k_rnaseq_results[run_label] = df


## kmeans ensemble plot

In [ ]:
import re
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from scipy.stats import mannwhitneyu, wilcoxon

mpl.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "xtick.labelsize": 10.5,
    "ytick.labelsize": 10.5,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 4,
    "ytick.major.size": 4,
})

candidate_names = [
    "best_k_gtex_CLAMP_base_kmeans_results_df",
    "best_k_gtex_CLAMP_full_kmeans_results_df",
    "best_k_gtex_PLIER_kmeans_results_df",
    "best_k_gtex_rnaseq_kmeans_results_df",
    "best_k_gtex_pca_kmeans_results_df",
    "best_k_gtex_ica_kmeans_results_df",
    "best_k_gtex_nmf_kmeans_results_df",
    "best_k_gtex_mofabc_kmeans_results_df",
    "best_k_gtex_flashier_kmeans_results_df",
    "best_k_gtex_GenomicSuperSignature_kmeans_results_df",
]

datasets = {name: globals().get(name) for name in candidate_names}
datasets = {
    k: v for k, v in datasets.items()
    if v is not None and "ari" in v.columns and len(v) > 0
}

if len(datasets) == 0:
    raise ValueError("No valid datasets found with an 'ari' column.")

def clean_name(name: str) -> str:
    name = re.sub(r"^best_k_gtex_", "", name)
    name = re.sub(r"_kmeans_results_df$", "", name)
    return name

display_name_map = {
    "CLAMP_base": "CLAMPbase",
    "CLAMP_full": "CLAMPfull",
    "PLIER": "PLIER",
    "rnaseq": "RNA-Seq",
    "mofabc": "MOFA-FLEX",
    "flashier": "Flashier",
    "GenomicSuperSignature": "GenomicSuperSignature",
    "pca": "PCA",
    "ica": "ICA",
    "nmf": "NMF",
}

def get_display_name(name: str) -> str:
    cleaned = clean_name(name)
    return display_name_map.get(cleaned, cleaned)

means = {
    name: np.asarray(datasets[name]["ari"], dtype=float).mean()
    for name in datasets
}

PINNED_ORDER = ["CLAMPfull", "PLIER", "CLAMPbase", "NMF"]

def order_key(name):
    display = get_display_name(name)
    if display in PINNED_ORDER:
        return (0, PINNED_ORDER.index(display))
    return (1, -means[name])

ordered_names = sorted(means, key=order_key)

data = [np.asarray(datasets[name]["ari"], dtype=float) for name in ordered_names]
labels = [get_display_name(name) for name in ordered_names]

def bh_adjust(pvals):
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)
    order = np.argsort(pvals)
    ranked = pvals[order]
    adj = np.empty(n, dtype=float)
    running = 1.0
    for i in range(n - 1, -1, -1):
        rank = i + 1
        val = ranked[i] * n / rank
        running = min(running, val)
        adj[order[i]] = min(running, 1.0)
    return adj

def mw_p_greater(a, b):
    return wilcoxon(a, b, alternative="greater").pvalue

def mw_p_two_sided(a, b):
    return wilcoxon(a, b, alternative="two-sided").pvalue

def format_q(q):
    if q >= 1e-3:
        return f"{q:.3f}"
    exponent = int(np.floor(np.log10(q)))
    coeff = q / (10 ** exponent)
    return rf"{coeff:.1f}$\times$10$^{{{exponent}}}$"

specific_pairs = [
    ("CLAMPfull", "PLIER"),
    ("CLAMPfull", "CLAMPbase"),
    ("CLAMPfull", "RNA-Seq"),
    ("CLAMPbase", "RNA-Seq"),
    ("CLAMPfull", "NMF"),
    ("CLAMPbase", "PLIER"),
    ("CLAMPbase", "NMF"),
]
available_pairs = [(a, b) for a, b in specific_pairs if a in labels and b in labels]

raw_pvals = [
    mw_p_greater(data[labels.index(a)], data[labels.index(b)])
    for a, b in available_pairs
]
qvals = bh_adjust(raw_pvals)
comparisons_to_draw = [(a, b, q) for (a, b), q in zip(available_pairs, qvals)]

raw_pvals_two_sided = [
    mw_p_two_sided(data[labels.index(a)], data[labels.index(b)])
    for a, b in available_pairs
]
qvals_two_sided = bh_adjust(raw_pvals_two_sided)

fig, ax = plt.subplots(figsize=(8.8, 5.6))
rng = np.random.default_rng(7)

positions = np.arange(1, len(data) + 1)

box = ax.boxplot(
    data,
    positions=positions,
    widths=0.52,
    patch_artist=True,
    showfliers=False,
    medianprops=dict(color="black", linewidth=1.4),
    whiskerprops=dict(color="black", linewidth=0.9),
    capprops=dict(color="black", linewidth=0.9),
    boxprops=dict(edgecolor="black", linewidth=0.9),
)

for patch, label in zip(box["boxes"], labels):
    patch.set_facecolor(MODEL_COLORS.get(label, "#BBBBBB"))
    patch.set_alpha(0.80)

for x, vals in zip(positions, data):
    jitter = rng.normal(0, 0.035, size=len(vals))
    ax.scatter(
        np.full(len(vals), x) + jitter,
        vals,
        s=18,
        facecolor="white",
        edgecolor="#2b2b2b",
        linewidth=0.55,
        alpha=0.85,
        zorder=3,
    )

for x, vals in zip(positions, data):
    mean_val = np.mean(vals)
    ymax = np.max(vals)
    ax.scatter(
        x, mean_val,
        s=40,
        marker="D",
        facecolor="white",
        edgecolor="black",
        linewidth=0.9,
        zorder=4,
    )
    ax.text(
        x, ymax + 0.018,
        f"{mean_val:.3f}",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
    )

ax.set_ylabel("Adjusted Rand index (ARI)")
ax.set_xlabel("")
ax.set_xticks(positions)
ax.set_xticklabels(labels, rotation=30, ha="right")
ax.set_xlim(0.45, len(data) + 0.55)
ax.set_ylim(0, 0.82)
ax.yaxis.set_major_locator(MultipleLocator(0.2))

ax.grid(axis="y", color="#D9D9D9", linewidth=0.7)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

def draw_bracket(ax, x1, x2, y, h, text):
    ax.plot(
        [x1, x1, x2, x2],
        [y, y + h, y + h, y],
        lw=0.9,
        c="black",
        clip_on=False,
    )
    ax.text(
        (x1 + x2) / 2,
        y + h + 0.008,
        text,
        ha="center",
        va="bottom",
        fontsize=9,
        clip_on=False,
    )

xpos = {lab: pos for lab, pos in zip(labels, positions)}

def assign_bracket_tiers(pairs_with_labels):
    rows = [
        {"a": a, "b": b, "left": l, "right": r, "q": q, "span": r - l}
        for a, b, l, r, q in pairs_with_labels
    ]
    rows.sort(key=lambda d: (d["span"], d["q"]))
    levels_used = []
    for row in rows:
        tier = 0
        while True:
            if tier >= len(levels_used):
                levels_used.append([])
                break
            if not any(not (row["right"] < lo or row["left"] > hi) for lo, hi in levels_used[tier]):
                break
            tier += 1
        row["tier"] = tier
        levels_used[tier].append((row["left"], row["right"]))
    return rows

pairs_with_labels = [
    (a, b, *sorted([xpos[a], xpos[b]]), q) for a, b, q in comparisons_to_draw
]
tier_by_pair = {(r["a"], r["b"]): r["tier"] for r in assign_bracket_tiers(pairs_with_labels)}

Y_BASE, Y_STEP, h = 0.845, 0.065, 0.028

for a, b, q in comparisons_to_draw:
    x1, x2 = sorted([xpos[a], xpos[b]])
    y = Y_BASE + tier_by_pair[(a, b)] * Y_STEP
    draw_bracket(ax, x1, x2, y, h, format_q(q))

fig.subplots_adjust(left=0.11, right=0.98, bottom=0.22, top=0.68)

plt.show()


In [ ]:
ARI_EXPORT_DIR = Path(here("output/03_model_biology/01_gtex/00_kmeans_clustering"))
ARI_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

ari_export_df = pd.DataFrame({
    "method": [lab for lab, vals in zip(labels, data) for _ in vals],
    "ari": [v for vals in data for v in vals],
})
ari_export_df.to_csv(ARI_EXPORT_DIR / "ari_data.csv", index=False)

ari_comparisons_df = pd.DataFrame({
    "a": [a for a, b in available_pairs],
    "b": [b for a, b in available_pairs],
    "p_raw": raw_pvals,
    "q": qvals,
    "p_raw_two_sided": raw_pvals_two_sided,
    "q_two_sided": qvals_two_sided,
})
ari_comparisons_df.to_csv(ARI_EXPORT_DIR / "ari_comparisons.csv", index=False)

print(f"Wrote {len(ari_export_df)} ARI rows across {ari_export_df['method'].nunique()} "
      f"methods and {len(ari_comparisons_df)} comparisons to {ARI_EXPORT_DIR}")


In [ ]:
fraction_order = [1.0, 0.75, 0.50, 0.25, 0.10, 0.05, 0.01]
fraction_labels = ["100%", "75%", "50%", "25%", "10%", "5%", "1%"]

def mw_p_two_sided(a, b):
    return mannwhitneyu(a, b, alternative="two-sided").pvalue

aggregated_data = {}
for frac, label in zip(fraction_order, fraction_labels):
    ari_values = []
    for run_label, df in best_k_rnaseq_results.items():
        if df["frac"].iloc[0] == frac:
            ari_values.extend(df["ari"].tolist())
    aggregated_data[label] = np.array(ari_values)

data = [aggregated_data[label] for label in fraction_labels]

fig, ax = plt.subplots(figsize=(8.8, 5.6))
rng = np.random.default_rng(7)

positions = np.arange(1, len(data) + 1)

box = ax.boxplot(
    data,
    positions=positions,
    widths=0.52,
    patch_artist=True,
    showfliers=False,
    medianprops=dict(color="black", linewidth=1.4),
    whiskerprops=dict(color="black", linewidth=0.9),
    capprops=dict(color="black", linewidth=0.9),
    boxprops=dict(edgecolor="black", linewidth=0.9),
)

for patch, label in zip(box["boxes"], fraction_labels):
    patch.set_facecolor(get_fraction_color(label))
    patch.set_alpha(0.80)

for x, vals in zip(positions, data):
    jitter = rng.normal(0, 0.035, size=len(vals))
    ax.scatter(
        np.full(len(vals), x) + jitter,
        vals,
        s=18,
        facecolor="white",
        edgecolor="#2b2b2b",
        linewidth=0.55,
        alpha=0.85,
        zorder=3,
    )

for x, vals in zip(positions, data):
    mean_val = np.mean(vals)
    ymax = np.max(vals)
    ax.scatter(
        x, mean_val,
        s=40,
        marker="D",
        facecolor="white",
        edgecolor="black",
        linewidth=0.9,
        zorder=4,
    )
    ax.text(
        x, ymax + 0.018,
        f"{mean_val:.3f}",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
    )

ax.set_ylabel("Adjusted Rand index (ARI)")
ax.set_xlabel("Percentage of genes used for RNA-Seq")
ax.set_xticks(positions)
ax.set_xticklabels(fraction_labels)
ax.set_xlim(0.45, len(data) + 0.55)

ax.grid(axis="y", color="#D9D9D9", linewidth=0.7)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

reference_label = "25%"
pairs = [(reference_label, label) for label in fraction_labels if label != reference_label]
raw_pvals = [mw_p_two_sided(aggregated_data[a], aggregated_data[b]) for a, b in pairs]
qvals = bh_adjust(raw_pvals)
comparisons_to_draw = [(a, b, q) for (a, b), q in zip(pairs, qvals)]

xpos = {label: i + 1 for i, label in enumerate(fraction_labels)}
pairs_with_labels = [
    (a, b, *sorted([xpos[a], xpos[b]]), q) for a, b, q in comparisons_to_draw
]
tier_by_pair = {(r["a"], r["b"]): r["tier"] for r in assign_bracket_tiers(pairs_with_labels)}

data_max = max(vals.max() for vals in data)
Y_BASE, Y_STEP, h = data_max + 0.06, 0.065, 0.028
n_tiers = max(tier_by_pair.values()) + 1
y_max = Y_BASE + (n_tiers - 1) * Y_STEP + h + 0.05

for a, b, q in comparisons_to_draw:
    x1, x2 = sorted([xpos[a], xpos[b]])
    y = Y_BASE + tier_by_pair[(a, b)] * Y_STEP
    draw_bracket(ax, x1, x2, y, h, format_q(q))

ax.set_ylim(0, y_max)
ax.yaxis.set_major_locator(MultipleLocator(0.2))

fig.subplots_adjust(left=0.11, right=0.98, bottom=0.15, top=0.95)

plt.show()


In [ ]:
gene_frac_export_df = pd.DataFrame({
    "fraction": [label for label, vals in zip(fraction_labels, data) for _ in vals],
    "ari": [v for vals in data for v in vals],
})
gene_frac_export_df.to_csv(ARI_EXPORT_DIR / "gene_fraction_ari_data.csv", index=False)

gene_frac_comparisons_df = pd.DataFrame(comparisons_to_draw, columns=["a", "b", "q"])
gene_frac_comparisons_df.to_csv(ARI_EXPORT_DIR / "gene_fraction_ari_comparisons.csv", index=False)

print(f"Wrote {len(gene_frac_export_df)} gene-fraction ARI rows across "
      f"{gene_frac_export_df['fraction'].nunique()} fractions and "
      f"{len(gene_frac_comparisons_df)} comparisons to {ARI_EXPORT_DIR}")
